# 05 · Fine-tune MMS Akan (VITS)
**T4 GPU runtime. Run the cells in order; if one fails, stop and paste its full error into the chat.** Each stage is
its own cell so a failure points at one step.

Uses the open-source recipe `ylacombe/finetune-hf-vits` (training loop and discriminator code are theirs; this
notebook and the repo's data preparation are ours). It needs older library versions than Colab ships (it was written
for transformers 4.35 to 4.37), so everything runs inside a clean pinned virtual environment, `/content/venv`, built with `uv`.
Training writes to Google Drive so a disconnect does not lose it.

## 1 · Environment (about 8 minutes, once per runtime)

In [ ]:
!test -d /content/tts && git -C /content/tts pull -q || git clone -q https://github.com/YCienne/TTS.git /content/tts
# Colab has no python3-venv, so use uv (no ensurepip needed; also much faster). It fetches Python 3.11 itself.
!pip install -q uv
!rm -rf /content/venv
!uv venv /content/venv --python 3.11 -q
!uv pip install --python /content/venv/bin/python -q torch==2.4.1 transformers==4.44.2 "datasets[audio]==2.21.0" accelerate==0.34.2 pyarrow==17.0.0 "numpy<2" soundfile librosa Cython setuptools wheel tensorboard matplotlib pandas scipy
!/content/venv/bin/python -c "import torch, transformers, datasets; print('torch', torch.__version__, '(built for CUDA', torch.version.cuda, ') cuda available:', torch.cuda.is_available(), '| transformers', transformers.__version__, '| datasets', datasets.__version__)"
# Colab sets MPLBACKEND to a notebook backend that is not installed in the clean venv; use a headless one.
%env MPLBACKEND=Agg
# Is there a GPU on this runtime at all? If not: Runtime > Change runtime type > T4 GPU, then run from the top.
!nvidia-smi -L || echo "NO GPU ON THIS RUNTIME"
from google.colab import drive
drive.mount('/content/drive')

## 2 · The training recipe, and its alignment extension

In [ ]:
%env MPLBACKEND=Agg
!test -d /content/finetune-hf-vits || git clone -q https://github.com/ylacombe/finetune-hf-vits /content/finetune-hf-vits
%cd /content/finetune-hf-vits/monotonic_align
!mkdir -p monotonic_align && /content/venv/bin/python setup.py build_ext --inplace
%cd /content/finetune-hf-vits
!/content/venv/bin/python -c "from utils.modeling_vits_training import VitsModelForPreTraining; print('recipe model class imports OK')"

## 3 · Convert Meta's Akan checkpoint into a *training* checkpoint
Downloads the generator and the discriminator (`D_100000.pth`, 561 MB) from `facebook/mms-tts` and joins them.

In [ ]:
%cd /content/finetune-hf-vits
!/content/venv/bin/python convert_original_discriminator_checkpoint.py --language_code aka --pytorch_dump_folder_path /content/mms-tts-aka-train
!ls -la /content/mms-tts-aka-train && cat /content/mms-tts-aka-train/vocab.json

## 4 · Choose the text variant
`PUNCT = True` adds `. , ? !` to the vocabulary (4 new embedding rows; punctuation then guides pauses and intonation).
`PUNCT = False` strips punctuation and uses the unmodified vocabulary: the safest first run.

In [ ]:
PUNCT = True
%cd /content/tts
if PUNCT:
    !/content/venv/bin/python scripts/extend_vocab.py --src /content/mms-tts-aka-train --dst /content/mms-tts-aka-punct --recipe /content/finetune-hf-vits
MODEL = "/content/mms-tts-aka-punct" if PUNCT else "/content/mms-tts-aka-train"

## 5 · Build the dataset (copies the prepared wavs from Drive to local disk)
Run `04_asr_check` first if you want the filtered metadata; this uses `metadata_filtered.tsv` when it exists.

In [ ]:
SPEAKER = "2"
DATA = f"/content/drive/MyDrive/akan_tts/data/speaker{SPEAKER}"
DS = "/content/aka_ds"
!rm -rf {DS}
flag = "" if PUNCT else "--strip-punct"
!/content/venv/bin/python /content/tts/scripts/make_audiofolder.py --data {DATA} --out {DS} {flag}
!/content/venv/bin/python -c "from datasets import load_dataset; d = load_dataset('{DS}'); print(d); print(d['train'][0]['text'], d['train'][0]['audio']['sampling_rate'])"

## 6 · Train
First a short smoke test (`EPOCHS = 2`) to prove the whole pipeline runs, then the real run. Output goes to Drive.
To resume after a disconnect set `RESUME = True` and re-run this cell.

In [ ]:
import json
EPOCHS = 2          # smoke test; use 100 for the real run
RESUME = False
RUN = f"/content/drive/MyDrive/akan_tts/runs/speaker{SPEAKER}_{'punct' if PUNCT else 'nopunct'}"
cfg = json.load(open("/content/tts/configs/train_aka.json"))
cfg.update(dataset_name=DS, model_name_or_path=MODEL, output_dir=RUN, num_train_epochs=EPOCHS,
           overwrite_output_dir=not RESUME)
if not PUNCT:
    cfg["full_generation_sample_text"] = cfg["full_generation_sample_text"]  # already has no punctuation
else:
    cfg["full_generation_sample_text"] = "mehunu mununkum. awia nso rebɔ kɛse pa ara."
if RESUME:
    cfg["resume_from_checkpoint"] = "latest"
json.dump(cfg, open("/content/run.json", "w"), ensure_ascii=False, indent=2)
print(RUN)

In [ ]:
%cd /content/finetune-hf-vits
!/content/venv/bin/accelerate launch run_vits_finetuning.py /content/run.json 2>&1 | tail -60

## 7 · Listen to the result (after the smoke test or any later run)

In [ ]:
PROMPTS = "mehunu mununkum. awia nso rebɔ kɛse pa ara."
!/content/venv/bin/python /content/tts/scripts/synthesize.py --model {RUN} --text "{PROMPTS}" --out /content/samples
import IPython.display as ipd
display(ipd.Audio("/content/samples/001.wav"))